## Fabric notebook source
 METADATA 
 
 Bronze — commit staged tables.
#
 The Copy activity lands raw rows in stg_<entity>. It cannot add audit
 columns, and Bronze rows without _batch_id cannot be re-run for a single
 window — any mistake would mean reloading everything. This notebook stamps
 the audit columns and appends staging into the real Bronze tables.
#
 It runs ONCE per pipeline, after the ForEach loop, and handles every staged
 entity in a single Spark session. The alternative — a notebook activity
 inside the loop — pays 10-30 seconds of session startup per entity, which
 across eleven tables is several minutes of doing nothing.
#
### Parameters (set by the pipeline):
-   batch_id : the batch to commit
-   entities : JSON array from the Lookup, so the notebook knows the staging -> target mapping without querying the warehouse


In [7]:
# batch_id = "MANUAL_RUN"
# entities = "[]"

StatementMeta(, 5f2e3a8b-64bc-4a06-8774-41ba43b1a9d8, 9, Finished, Available, Finished, False)

In [9]:
%run 00_common_utils

StatementMeta(, 5f2e3a8b-64bc-4a06-8774-41ba43b1a9d8, 11, Finished, Available, Finished, True)

In [10]:
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

run_ts = datetime.now(timezone.utc)
entity_list = json.loads(entities) if isinstance(entities, str) else entities

if not entity_list:
    raise ValueError(
        "No entities passed. The pipeline must forward the Lookup output as "
        "the 'entities' parameter."
    )

# All table names are UNQUALIFIED and resolve against the attached default
# lakehouse. Hardcoding a "lh_bronze." prefix ties the notebook to one
# lakehouse name; when the item was actually called h1_bronze, every
# spark.table() call raised AnalysisException, the loop swallowed it as
# "no staging table", and eleven entities silently committed nothing.
existing = {t.tableName for t in spark.sql("SHOW TABLES").collect()}
print(f"Committing batch {batch_id} — {len(entity_list)} entities")
print(f"Tables visible in default lakehouse: {len(existing)}")
print(f"Staging tables found: {sorted(t for t in existing if t.startswith('stg_'))}\n")

StatementMeta(, 5f2e3a8b-64bc-4a06-8774-41ba43b1a9d8, 12, Finished, Available, Finished, False)

Committing batch SQLSRV_20260804054930_6834f885 — 11 entities


## Commit each staged table

 The audit columns are the point of this step:

  - _source_system  which system it came from
  - _batch_id       which run loaded it, so one window can be replayed
  - _ingest_ts      when it landed
  - _load_date      partition key
  - _row_hash       payload fingerprint, used by Silver to deduplicate

 The hash covers only the payload columns, never the audit columns. Hashing
 _ingest_ts would make every row unique on every run and defeat the
 deduplication it exists to enable.

In [11]:
results = []

for ent in entity_list:
    source_system = ent["source_system"]
    entity_name   = ent["entity_name"]
    target_table  = ent["target_table"]
    load_type     = ent.get("load_type", "incremental")

    staging_table = f"stg_{target_table}"
    bronze_table  = target_table
    label = f"{source_system}.{entity_name}"

    # API entities now arrive as JSON files rather than staging tables, so
    # the table-existence check does not apply to them.
    if source_system not in ("PHARM", "FACIL") and staging_table not in existing:
        # A copy that matched no rows may not create a staging table.
        # Normal incremental outcome, not a failure.
        print(f"  {label:<34} no staging table — 0 rows")
        results.append({"entity": label, "target": target_table,
                        "rows": 0, "status": "NO_DATA"})
        continue

    if source_system in ("PHARM", "FACIL"):
        raw = spark.read.option("multiLine", True).json(
            f"Files/landing/API/{staging_table}*.jaon")
        stg = raw.select(F.explode("data").alias("r")).select("r.*")
    else:
        stg = spark.table(staging_table)
    # The REST connector's collection reference does not persist reliably in
    # this workspace, so the whole response envelope lands as columns with
    # literal dots in their names: data.bed_id, pagination.next_url,
    # meta.server_time. Flatten here instead — the notebook is version
    # controlled and testable, the connector setting is neither.
    #
    # Backticks are required. Without them Spark reads "data.bed_id" as
    # field bed_id inside struct data, which does not exist, and the error
    # names a column that looks nothing like the real problem.
    if any(c.startswith("data.") for c in stg.columns):
        stg = stg.select([
            F.col(f"`{c}`").alias(c[len("data."):])
            for c in stg.columns
            if c.startswith("data.")
        ])
        print(f"  {label:<34} flattened REST envelope "
              f"({len(stg.columns)} columns kept)")

    row_count = stg.count()

    if row_count == 0:
        print(f"  {label:<34} staging empty — 0 rows")
        spark.sql(f"DROP TABLE IF EXISTS {staging_table}")
        results.append({"entity": label, "target": target_table,
                        "rows": 0, "status": "NO_DATA"})
        continue

    payload_cols = [c for c in stg.columns if not c.startswith("_")]

    enriched = (
        stg
        .withColumn("_source_system", F.lit(source_system))
        .withColumn("_entity_name",   F.lit(entity_name))
        .withColumn("_batch_id",      F.lit(batch_id))
        .withColumn("_ingest_ts",     F.lit(run_ts))
        .withColumn("_load_date",     F.lit(run_ts.date()))
    )
    # Hash the payload only. Including _ingest_ts would make every row
    # unique on every run and defeat the deduplication it exists to enable.
    enriched = row_hash(enriched, payload_cols)

    # full_snapshot re-reads everything each run, so a plain append would
    # duplicate the table daily. Replacing just today's partition keeps
    # Bronze append-only across days while staying idempotent within a day.
    if load_type == "full_snapshot" and bronze_table in existing:
        (enriched.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"_load_date = '{run_ts.date()}'")
            .option("mergeSchema", "true")
            .partitionBy("_load_date")
            .saveAsTable(bronze_table))
        mode_used = "replaceWhere"
    else:
        (enriched.write.format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .partitionBy("_load_date")
            .saveAsTable(bronze_table))
        mode_used = "append"

    spark.sql(f"DROP TABLE IF EXISTS {staging_table}")

    print(f"  {label:<34} {row_count:>9,} rows  ({mode_used})")
    results.append({"entity": label, "target": target_table,
                    "rows": row_count, "status": "COMMITTED"})

StatementMeta(, 5f2e3a8b-64bc-4a06-8774-41ba43b1a9d8, 13, Finished, Available, Finished, False)

  EHR.admission                      no staging table — 0 rows
  EHR.bed_assignment                 no staging table — 0 rows
  EHR.diagnosis                      no staging table — 0 rows
  EHR.emergency_visit                no staging table — 0 rows
  EHR.patient                        no staging table — 0 rows
  FIN.invoice                        no staging table — 0 rows
  FIN.invoice_line                   no staging table — 0 rows
  FIN.patient_account                no staging table — 0 rows
  SCHED.appointment                    no staging table — 0 rows
  SCHED.appointment_status_history     no staging table — 0 rows
  SCHED.patient                        no staging table — 0 rows


## Verify before reporting success

The pipeline advances watermarks based on this notebook succeeding. If it
reports success without data actually reaching Bronze, the watermark moves
past a window that was never loaded, and the gap is invisible.

In [12]:
committed  = [r for r in results if r["status"] == "COMMITTED"]
no_data    = [r for r in results if r["status"] == "NO_DATA"]
total_rows = sum(r["rows"] for r in results)

print(f"\nCommitted : {len(committed)} entities, {total_rows:,} rows")
print(f"No data   : {len(no_data)} entities")

# The pipeline advances watermarks on this notebook succeeding. Reporting
# success without data reaching Bronze moves the watermark past a window
# that was never loaded, and the gap leaves no trace.
mismatches = []
for r in committed:
    in_bronze = (spark.table(r["target"])
                      .filter(F.col("_batch_id") == batch_id)
                      .count())
    if in_bronze != r["rows"]:
        mismatches.append(f"{r['entity']}: staged {r['rows']:,} "
                          f"but {in_bronze:,} in Bronze")
if mismatches:
    raise RuntimeError("Row counts do not reconcile:\n  " + "\n  ".join(mismatches))
if committed:
    print("Reconciliation: staging and Bronze row counts match for every entity")

leftover = sorted(t.tableName for t in spark.sql("SHOW TABLES").collect()
                  if t.tableName.startswith("stg_"))
if leftover:
    raise RuntimeError(
        f"Staging tables still present after commit: {leftover}. "
        f"These were copied but not committed — their rows are invisible "
        f"while the watermark is about to advance past them.")

mssparkutils.notebook.exit(json.dumps({
    "batch_id": batch_id,
    "entities_committed": len(committed),
    "total_rows": total_rows,
}))

StatementMeta(, 5f2e3a8b-64bc-4a06-8774-41ba43b1a9d8, 14, Finished, Available, Finished, False)


Committed : 0 entities, 0 rows
No data   : 11 entities


RuntimeError: Staging tables still present after commit: ['stg_ehr_admission', 'stg_ehr_bed_assignment', 'stg_ehr_diagnosis', 'stg_ehr_emergency_visit', 'stg_ehr_patient', 'stg_fin_invoice', 'stg_fin_invoice_line', 'stg_fin_patient_account', 'stg_sched_appointment', 'stg_sched_appointment_status_history', 'stg_sched_patient']. These were copied but not committed — check the entities parameter matches the Lookup output.